<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Data Carpentry · Python profesional</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Soluciones — Clean code, estructura de proyecto, nombres e imports**

Este laboratorio trabaja sobre una idea central: **que el código funcione no implica que esté bien diseñado**.

Seguiremos siempre el patrón:

> **ver el problema → diagnosticarlo → refactorizarlo → validar la mejora**

## Objetivos de aprendizaje

Al finalizar deberías poder reconocer y corregir nombres ambiguos, números mágicos, duplicación, funciones con demasiadas responsabilidades, estructuras de proyecto pobres, imports problemáticos y dependencias circulares. También crearás físicamente un pequeño paquete Python desde el propio notebook.

## Cómo trabajar

**Solución:** conserva la parte guiada e incluye doce propuestas resueltas, con comprobaciones de equivalencia, imports y exportaciones. Las respuestas de diseño son orientativas.

**Entorno:** Python, NumPy, Pandas y Matplotlib. Trabaja desde la carpeta de la sesión. Se crean áreas nuevas en `demos_clean_code/`; las soluciones añaden `practicas_clean_code/` y CSV de demostración. Los errores de imports y circularidad son deliberados y se capturan. Selecciona con el docente las actividades de aula; el resto puede servir de ampliación.


## 1. Preparación del entorno

In [ ]:
from pathlib import Path
import sys
import importlib
import shutil
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED=42
rng=np.random.default_rng(SEED)
print("Entorno preparado")

## 2. Problema 1 — Nombres ambiguos

Primero veremos una versión que funciona pero obliga al lector a reconstruir el contexto.

In [ ]:
x = pd.DataFrame({
    "a": rng.integers(1, 8, 20),
    "b": rng.uniform(5, 100, 20),
    "c": rng.choice(["A", "B"], 20)
})
y = x["a"] * x["b"]
z = y.mean()
x.head(), z

### Diagnóstico

¿Qué significan `x`, `a`, `b`, `y` y `z`? El código es correcto sintácticamente pero su carga cognitiva es alta. La intención no está en el código.

## 2.1 Refactorización de nombres

In [ ]:
ventas = x.rename(columns={"a": "unidades", "b": "precio_unitario", "c": "canal"}).copy()
importe_venta = ventas["unidades"] * ventas["precio_unitario"]
importe_medio = importe_venta.mean()
np.testing.assert_allclose(importe_venta, y)
np.testing.assert_allclose(importe_medio, z)
ventas.head(), importe_medio


### Interpretación

La lógica es la misma, pero el lector ya no necesita recordar qué representa cada objeto. Un buen nombre reduce contexto externo.

## 3. Problema 2 — Números mágicos

In [ ]:
ventas_demo = pd.DataFrame({"importe": rng.gamma(2, 150, 50)})
ventas_demo["tipo"] = np.where(ventas_demo["importe"] > 500, "alta", "normal")
ventas_demo.head()

### Diagnóstico

`500` puede ser una regla de negocio, un valor provisional o una decisión normativa. El código no lo explica.

In [ ]:
UMBRAL_VENTA_ALTA = 500
ventas_demo["tipo"] = np.where(
    ventas_demo["importe"] > UMBRAL_VENTA_ALTA,
    "alta",
    "normal"
)
ventas_demo.head()

### Validación de la mejora

El valor sigue siendo el mismo, pero ahora la intención es explícita y el cambio está centralizado.

## 4. Problema 3 — Duplicación

In [ ]:
clientes_a = pd.DataFrame({"ingresos": [32000, 42000, 28000]})
clientes_b = pd.DataFrame({"ingresos": [51000, 26000, 39000]})
clientes_a["ingresos_mensuales"] = clientes_a["ingresos"] / 12
clientes_a["ingresos_altos"] = clientes_a["ingresos"] > 40000
clientes_b["ingresos_mensuales"] = clientes_b["ingresos"] / 12
clientes_b["ingresos_altos"] = clientes_b["ingresos"] > 40000
clientes_a, clientes_b

### Diagnóstico

La misma regla existe en dos lugares. Si mañana cambia el umbral, una copia podría actualizarse y la otra no.

In [ ]:
def enriquecer_clientes(datos: pd.DataFrame, umbral_ingresos_altos: float=40000) -> pd.DataFrame:
    salida=datos.copy()
    salida["ingresos_mensuales"] = salida["ingresos"] / 12
    salida["ingresos_altos"] = salida["ingresos"] > umbral_ingresos_altos
    return salida
clientes_a_ref=enriquecer_clientes(clientes_a[["ingresos"]])
clientes_b_ref=enriquecer_clientes(clientes_b[["ingresos"]])
clientes_a_ref, clientes_b_ref

### Interpretación

La transformación tiene ahora una única fuente de verdad. Esto aplica DRY cuando existe una misma intención conceptual.

## 5. Problema 4 — Función monolítica

In [ ]:
def proceso_monolitico(datos: pd.DataFrame) -> pd.DataFrame:
    salida=datos.copy()
    salida=salida.drop_duplicates()
    salida["importe"]=salida["unidades"]*salida["precio"]
    salida["venta_alta"]=salida["importe"]>500
    resumen=(salida.groupby("canal").agg(
        operaciones=("importe","size"),
        importe_medio=("importe","mean")
    ).reset_index())
    print(resumen)
    resumen.to_csv("resumen_temporal.csv", index=False)
    return resumen

### Diagnóstico

La función limpia, transforma, clasifica, agrega, imprime y exporta. Tiene demasiadas responsabilidades y por ello es difícil de probar o reutilizar parcialmente.

## 5.1 Separar responsabilidades

In [ ]:
def eliminar_duplicados(datos):
    return datos.drop_duplicates().copy()

def calcular_importe(datos):
    salida=datos.copy(); salida["importe"]=salida["unidades"]*salida["precio"]; return salida

def clasificar_ventas(datos, umbral=500):
    salida=datos.copy(); salida["venta_alta"]=salida["importe"]>umbral; return salida

def resumir_por_canal(datos):
    return datos.groupby("canal").agg(
        operaciones=("importe","size"),
        importe_medio=("importe","mean")
    ).reset_index()

## 5.2 Ejecutar y validar la versión modular

In [ ]:
ventas_base = pd.DataFrame({
    "unidades": rng.integers(1,8,200),
    "precio": rng.uniform(10,200,200),
    "canal": rng.choice(["web","tienda","partner"],200)
})
ventas_mod=eliminar_duplicados(ventas_base)
ventas_mod=calcular_importe(ventas_mod)
ventas_mod=clasificar_ventas(ventas_mod)
resumen_modular=resumir_por_canal(ventas_mod)
resumen_modular

### Interpretación

Cada función tiene ahora una responsabilidad principal. Eso mejora cohesión y reduce el acoplamiento entre tareas.

# 6. Crear automáticamente la estructura real del proyecto

In [ ]:
import tempfile
base_demos = Path("demos_clean_code")
base_demos.mkdir(exist_ok=True)
raiz = Path(tempfile.mkdtemp(prefix="proyecto_", dir=base_demos)).resolve()
carpetas = [raiz / "data/raw", raiz / "data/processed", raiz / "notebooks", raiz / "src/ventas_project", raiz / "tests"]
for carpeta in carpetas:
    carpeta.mkdir(parents=True, exist_ok=True)
print("Estructura creada sin borrar proyectos previos:", raiz)


## 6.1 Crear ficheros mínimos

In [ ]:
ficheros={
    raiz/"src"/"ventas_project"/"__init__.py":"",
    raiz/"README.md":"# Proyecto Clean Code\n",
    raiz/".gitignore":".venv/\n__pycache__/\n.ipynb_checkpoints/\n",
    raiz/"tests"/"__init__.py":""
}
for ruta,contenido in ficheros.items():
    ruta.write_text(contenido,encoding="utf-8")
print("Ficheros creados")

## 6.2 Inspeccionar lo creado

In [ ]:
for ruta in sorted(raiz.rglob("*")):
    nivel=len(ruta.relative_to(raiz).parts)-1
    sangria="    "*nivel
    simbolo="[DIR]" if ruta.is_dir() else "[FILE]"
    print(f"{sangria}{simbolo} {ruta.name}")

## 6.3 Validar programáticamente la estructura

In [ ]:
estado={str(carpeta):carpeta.exists() for carpeta in carpetas}
pd.Series(estado, name="existe")

### Interpretación

No damos por supuesto que la estructura existe: la verificamos. Esto convierte una instrucción teórica en un procedimiento reproducible.

# 7. Crear módulos Python reales desde el notebook

In [ ]:
validation_code = """
import pandas as pd

def validar_columnas(datos: pd.DataFrame, requeridas: list[str]) -> None:
    faltantes=[c for c in requeridas if c not in datos.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
"""
features_code = """
import pandas as pd

def calcular_importe(datos: pd.DataFrame) -> pd.DataFrame:
    salida=datos.copy()
    salida["importe"]=salida["unidades"]*salida["precio"]
    return salida
"""
analysis_code = """
import pandas as pd

def resumir_por_canal(datos: pd.DataFrame) -> pd.DataFrame:
    return datos.groupby("canal").agg(
        operaciones=("importe","size"),
        importe_medio=("importe","mean")
    ).reset_index()
"""

In [ ]:
src_pkg=raiz/"src"/"ventas_project"
(src_pkg/"validation.py").write_text(validation_code,encoding="utf-8")
(src_pkg/"features.py").write_text(features_code,encoding="utf-8")
(src_pkg/"analysis.py").write_text(analysis_code,encoding="utf-8")
for archivo in sorted(src_pkg.glob("*.py")):
    print("-",archivo.name)

## 7.1 Verificar el contenido de un módulo

In [ ]:
print((src_pkg/"features.py").read_text(encoding="utf-8"))

### Interpretación

Las funciones ya existen fuera del notebook. Pueden versionarse, importarse y probarse de forma independiente.

# 8. Imports — primero veremos el fallo

In [ ]:
try:
    import ventas_project
except ModuleNotFoundError as error:
    print("Import fallido:")
    print(error)

### Diagnóstico

El paquete existe en disco, pero Python todavía no conoce `src/`. Existencia física e importabilidad son conceptos distintos.

## 8.1 Corregir temporalmente el path

In [ ]:
ruta_src=str((raiz/"src").resolve())
if ruta_src not in sys.path:
    sys.path.insert(0,ruta_src)
print(ruta_src)

## 8.2 Importar las funciones propias

In [ ]:
from ventas_project.validation import validar_columnas
from ventas_project.features import calcular_importe as calcular_importe_modulo
from ventas_project.analysis import resumir_por_canal as resumir_por_canal_modulo
print("Imports correctos")

## 8.3 Ejecutarlas

In [ ]:
datos_demo=pd.DataFrame({
    "unidades":[2,4,1],
    "precio":[10.0,7.5,30.0],
    "canal":["web","tienda","web"]
})
validar_columnas(datos_demo,["unidades","precio","canal"])
datos_demo=calcular_importe_modulo(datos_demo)
resumir_por_canal_modulo(datos_demo)

### Interpretación

El notebook ya se comporta como consumidor de código reusable: importa, ejecuta, visualiza e interpreta.

# 9. Imports problemáticos: `import *`

In [ ]:
print("from pandas import *")

### Diagnóstico

`import *` dificulta saber qué nombres entran en el namespace y aumenta colisiones. Preferimos imports explícitos.

## 9.1 Colisión de nombres — verla antes de corregirla

In [ ]:
from math import sqrt
sqrt_original=sqrt(16)
def sqrt(x):
    return "nombre sobrescrito"
sqrt_sobrescrito=sqrt(16)
sqrt_original, sqrt_sobrescrito

## 9.2 Corregir la colisión

In [ ]:
import math
math.sqrt(16)

### Interpretación

`math.sqrt` hace explícito el origen y reduce ambigüedad.

# 10. Imports absolutos y relativos

In [ ]:
print("Absoluto: from ventas_project.features import calcular_importe")
print("Relativo: from .features import calcular_importe")

### Interpretación

Los imports absolutos son muy explícitos; los relativos pueden ser útiles dentro de un paquete. Lo importante es mantener consistencia y claridad.

# 11. `__init__.py` y API pública

In [ ]:
init_code="""
from .features import calcular_importe
from .analysis import resumir_por_canal
"""
(src_pkg/"__init__.py").write_text(init_code,encoding="utf-8")
print((src_pkg/"__init__.py").read_text(encoding="utf-8"))

In [ ]:
import ventas_project
importlib.reload(ventas_project)
from ventas_project import calcular_importe, resumir_por_canal
print("API pública cargada")

### Interpretación

`__init__.py` puede exponer una interfaz cómoda y ocultar detalles internos del paquete.

# 12. Dependencia circular — provocar el problema

In [ ]:
circular_dir=raiz/"src"/"circular_demo"
circular_dir.mkdir(exist_ok=True)
(circular_dir/"__init__.py").write_text("",encoding="utf-8")
(circular_dir/"a.py").write_text("""
from .b import funcion_b

def funcion_a():
    return "A"
""",encoding="utf-8")
(circular_dir/"b.py").write_text("""
from .a import funcion_a

def funcion_b():
    return "B"
""",encoding="utf-8")
print("Dependencia circular creada")

## 12.1 Intentar importar

In [ ]:
try:
    import circular_demo.a
except Exception as error:
    print(type(error).__name__)
    print(error)

### Diagnóstico

Una dependencia circular suele señalar acoplamiento excesivo o una mala división de responsabilidades.

## 12.2 Refactorizar la lógica común

In [ ]:
(circular_dir/"common.py").write_text("""
def mensaje_base():
    return "Base común"
""",encoding="utf-8")
(circular_dir/"a.py").write_text("""
from .common import mensaje_base

def funcion_a():
    return f"A usa: {mensaje_base()}"
""",encoding="utf-8")
(circular_dir/"b.py").write_text("""
from .common import mensaje_base

def funcion_b():
    return f"B usa: {mensaje_base()}"
""",encoding="utf-8")
print("Circularidad eliminada")

## 12.3 Validar la corrección

In [ ]:
for nombre in ["circular_demo.a","circular_demo.b"]:
    sys.modules.pop(nombre,None)
from circular_demo.a import funcion_a
from circular_demo.b import funcion_b
print(funcion_a())
print(funcion_b())

### Interpretación

Extraer la dependencia común reduce acoplamiento y mejora cohesión.

# 13. Mini-práctica guiada — De script monolítico a paquete

In [ ]:
script_malo="""
import pandas as pd

df = pd.read_csv("ventas.csv")
df = df.drop_duplicates()
df["importe"] = df["unidades"] * df["precio"]
df["tipo"] = "normal"
df.loc[df["importe"] > 500, "tipo"] = "alta"
res = df.groupby("canal")["importe"].mean()
print(res)
res.to_csv("resumen.csv")
"""
print(script_malo)

### Diagnóstico

Ruta hardcodeada, ejecución global, nombres ambiguos, número mágico y mezcla de carga, limpieza, features, análisis y exportación.

## 13.1 Diseñar arquitectura

In [ ]:
print("""src/ventas_project/
├── io.py
├── cleaning.py
├── features.py
├── analysis.py
└── pipeline.py""")

## 13.2 Generar módulos automáticamente

In [ ]:
modulos_refactor={
"io.py":"""import pandas as pd

def cargar_csv(ruta):
    return pd.read_csv(ruta)
""",
"cleaning.py":"""def limpiar_duplicados(datos):
    return datos.drop_duplicates().copy()
""",
"features.py":"""def crear_importe(datos):
    salida=datos.copy()
    salida["importe"]=salida["unidades"]*salida["precio"]
    return salida
""",
"analysis.py":"""def resumen_canal(datos):
    return datos.groupby("canal").agg(operaciones=("importe","size"), importe_medio=("importe","mean")).reset_index()
""",
"pipeline.py":"""from .cleaning import limpiar_duplicados
from .features import crear_importe
from .analysis import resumen_canal

def ejecutar_pipeline(datos):
    datos=limpiar_duplicados(datos)
    datos=crear_importe(datos)
    return resumen_canal(datos)
"""
}
for nombre,contenido in modulos_refactor.items():
    (src_pkg/nombre).write_text(contenido,encoding="utf-8")
print("Módulos generados")

## 13.3 Inspeccionar los módulos

In [ ]:
for archivo in sorted(src_pkg.glob("*.py")):
    print("\n---",archivo.name,"---")
    print(archivo.read_text(encoding="utf-8"))

# 14. Visualizar la arquitectura conceptual antes y después

In [ ]:
categorias=["Carga","Limpieza","Features","Análisis","Exportación"]
monolitico=[1,1,1,1,1]
modular=[0,0,0,0,0]
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(len(categorias)); ancho=.35
ax.bar(x-ancho/2,monolitico,ancho,label="Script monolítico")
ax.bar(x+ancho/2,modular,ancho,label="Pipeline orquestador")
ax.set_xticks(x); ax.set_xticklabels(categorias,rotation=30)
ax.set_ylabel("Responsabilidad implementada directamente")
ax.set_title("Concentración de responsabilidades")
ax.legend(); plt.show()

### Interpretación del gráfico

La visualización es conceptual: el script monolítico implementa directamente todas las responsabilidades. En la versión modular, el orquestador delega cada tarea a un módulo especializado.

# 15. Checklist de clean code

- [ ] Los nombres comunican intención.
- [ ] No hay números mágicos sin justificar.
- [ ] No hay duplicación conceptual innecesaria.
- [ ] Cada función tiene una responsabilidad clara.
- [ ] La estructura separa datos, notebooks, código y tests.
- [ ] La lógica reusable vive en `src/`.
- [ ] Los imports son explícitos.
- [ ] No hay colisiones de nombres.
- [ ] No hay dependencias circulares.
- [ ] La API pública del paquete es clara.

# 16. Ejercicios finales


## Ejercicio 1 — Nombres

Refactoriza los nombres de `datos_ambiguos`, sus columnas `a`, `b`, `c` y los resultados `y_original` y `z_original` sin regenerar los datos. Comprueba que renombrar mantiene el importe y la media.


In [ ]:
datos_ambiguos = pd.DataFrame({'a': [2, 3, 1], 'b': [10.0, 20.0, 15.0], 'c': ['web', 'tienda', 'web']})
y_original = datos_ambiguos['a'] * datos_ambiguos['b']
z_original = y_original.mean()
ventas_nombres = datos_ambiguos.rename(columns={"a": "unidades", "b": "precio_unitario", "c": "canal"}).copy()
importe_por_venta = ventas_nombres["unidades"] * ventas_nombres["precio_unitario"]
importe_medio_ventas = importe_por_venta.mean()
np.testing.assert_allclose(importe_por_venta, y_original)
np.testing.assert_allclose(importe_medio_ventas, z_original)
print(ventas_nombres.head())
print("Importe medio:", importe_medio_ventas)


### Solución razonada

Cambiar nombres conserva el significado y el comportamiento. Usar una nueva extracción del generador aleatorio cambiaría las observaciones y no demostraría equivalencia de la refactorización.


## Ejercicio 2 — Números mágicos

Clasifica 12 meses/año, un umbral de venta alta de 500 y la ruta de exportación como constante, parámetro o configuración. Implementa una clasificación parametrizable y comprueba los casos 499, 500 y 501.


In [ ]:
MESES_POR_ANIO = 12
configuracion_practica = {"ruta_salida": "resultados/resumen.csv"}

def clasificar_importes(importes, umbral=500):
    return np.where(np.asarray(importes) > umbral, "alta", "normal")

np.testing.assert_array_equal(clasificar_importes([499, 500, 501]), ["normal", "normal", "alta"])
np.testing.assert_array_equal(clasificar_importes([501], umbral=600), ["normal"])
print("Límite estricto y parámetro comprobados.")


### Solución razonada

12 meses/año es una constante del dominio; el umbral es un parámetro de negocio cuyo valor por defecto puede centralizarse; la ruta depende del entorno y pertenece a configuración. Clasificar un valor no obliga a usar siempre el mismo mecanismo: el contexto determina dónde vive.


## Ejercicio 3 — Duplicación

Muestra dos bloques de enriquecimiento de clientes y sustitúyelos por una función. Calcula ingresos mensuales y una marca de ingresos altos (> 40.000). Verifica equivalencia y ausencia de mutación.


In [ ]:
grupo_a = pd.DataFrame({"ingresos": [32000, 42000, 28000]})
grupo_b = pd.DataFrame({"ingresos": [51000, 26000, 39000]})
original_a = grupo_a.copy(deep=True)
original_b = grupo_b.copy(deep=True)
referencia_a = grupo_a.copy()
referencia_a["ingresos_mensuales"] = referencia_a["ingresos"] / 12
referencia_a["ingresos_altos"] = referencia_a["ingresos"] > 40000
referencia_b = grupo_b.copy()
referencia_b["ingresos_mensuales"] = referencia_b["ingresos"] / 12
referencia_b["ingresos_altos"] = referencia_b["ingresos"] > 40000

def enriquecer_practica(datos, umbral=40000):
    salida = datos.copy()
    salida["ingresos_mensuales"] = salida["ingresos"] / MESES_POR_ANIO
    salida["ingresos_altos"] = salida["ingresos"] > umbral
    return salida

pd.testing.assert_frame_equal(enriquecer_practica(grupo_a), referencia_a)
pd.testing.assert_frame_equal(enriquecer_practica(grupo_b), referencia_b)
pd.testing.assert_frame_equal(grupo_a, original_a)
pd.testing.assert_frame_equal(grupo_b, original_b)
print("Ambos grupos conservan resultados y entradas.")


### Solución razonada

La abstracción se justifica porque los dos bloques implementan la misma regla. DRY evita múltiples fuentes de verdad; no exige fusionar tareas distintas solo porque algunas líneas se parecen.


## Ejercicio 4 — Responsabilidad única

Separa limpieza, cálculo de importe, resumen y exportación. Compara con la función monolítica de la parte guiada y comprueba que el CSV contiene el mismo resumen.


In [ ]:
def limpiar_practica(datos):
    return datos.drop_duplicates().copy()

def preparar_practica(datos):
    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]
    return salida

def resumir_practica(datos):
    return datos.groupby("canal", as_index=False).agg(
        operaciones=("importe", "size"), importe_medio=("importe", "mean"))

def exportar_practica(resumen, ruta):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    resumen.to_csv(ruta, index=False)

entrada_responsabilidades = pd.concat([ventas_base, ventas_base.iloc[[0]]], ignore_index=True)
entrada_copia = entrada_responsabilidades.copy(deep=True)
referencia_responsabilidades = proceso_monolitico(entrada_responsabilidades)
resumen_responsabilidades = resumir_practica(preparar_practica(limpiar_practica(entrada_responsabilidades)))
pd.testing.assert_frame_equal(resumen_responsabilidades, referencia_responsabilidades)
pd.testing.assert_frame_equal(entrada_responsabilidades, entrada_copia)
ruta_csv_practica = Path("resultados_clean_code") / "resumen.csv"
exportar_practica(resumen_responsabilidades, ruta_csv_practica)
pd.testing.assert_frame_equal(pd.read_csv(ruta_csv_practica), resumen_responsabilidades)
print("Resumen y exportación equivalentes.")


### Solución razonada

El cálculo de venta_alta de la función original no participa en su resumen: puede omitirse para este contrato de salida. Si otro consumidor necesitara esa columna, habría que mantener una etapa explícita de clasificación. La exportación es una responsabilidad separada y tiene efectos sobre disco.


## Ejercicio 5 — Crear estructura automáticamente

Crea data/raw, data/processed, notebooks, src, tests y README.md con pathlib. Usa una carpeta nueva dentro de `practicas_clean_code` y comprueba existencia e idempotencia sin borrar archivos anteriores.


In [ ]:
import tempfile
base_practicas = Path("practicas_clean_code")
base_practicas.mkdir(exist_ok=True)
raiz_practica = Path(tempfile.mkdtemp(prefix="ventas_", dir=base_practicas)).resolve()
subcarpetas_practica = ["data/raw", "data/processed", "notebooks", "src/ventas_practica", "tests"]
for _ in range(2):
    for relativa in subcarpetas_practica:
        (raiz_practica / relativa).mkdir(parents=True, exist_ok=True)
(raiz_practica / "README.md").write_text("# Proyecto de ventas de práctica\n", encoding="utf-8")
assert all((raiz_practica / relativa).is_dir() for relativa in subcarpetas_practica)
assert (raiz_practica / "README.md").is_file()
print("Estructura verificada:", raiz_practica)


### Solución razonada

Cada ejecución completa crea un área nueva para no sobrescribir trabajos anteriores. Dentro de esa área, mkdir con exist_ok=True puede repetirse. La existencia de carpetas no demuestra que el diseño sea adecuado, pero verifica que la creación se realizó.


## Ejercicio 6 — Crear módulos

Genera validation.py, features.py y analysis.py dentro del paquete del ejercicio 5. La validación exige columnas y valores positivos y finitos de unidades y precio. Comprueba sintaxis de los módulos.


In [ ]:
paquete_practica = raiz_practica / "src" / "ventas_practica"
fuentes_practica = {
    "__init__.py": "",
    "validation.py": """import numpy as np
import pandas as pd

def validar_ventas(datos):
    requeridas = {"unidades", "precio", "canal"}
    faltantes = requeridas - set(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas: {sorted(faltantes)}")
    if datos[list(requeridas)].isna().any().any():
        raise ValueError("Hay valores nulos en columnas obligatorias.")
    for columna in ["unidades", "precio"]:
        if not pd.api.types.is_numeric_dtype(datos[columna]) or pd.api.types.is_bool_dtype(datos[columna]):
            raise ValueError(f"{columna} debe ser numérica y no booleana.")
        if not np.isfinite(datos[columna].to_numpy(dtype=float)).all():
            raise ValueError(f"{columna} debe contener valores finitos.")
        if (datos[columna] <= 0).any():
            raise ValueError(f"{columna} debe ser positiva.")
""",
    "features.py": """def preparar_ventas(datos, umbral=500):
    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]
    salida["venta_alta"] = salida["importe"] > umbral
    return salida
""",
    "analysis.py": """def resumir_ventas(datos):
    return datos.groupby("canal", as_index=False).agg(
        operaciones=("importe", "size"), importe_medio=("importe", "mean"))
""",
}
for nombre, fuente in fuentes_practica.items():
    compile(fuente, nombre, "exec")
    (paquete_practica / nombre).write_text(fuente, encoding="utf-8")
print("Módulos escritos y sintaxis comprobada.")


### Solución razonada

La validación depende del contrato acordado: aquí las unidades y el precio deben ser estrictamente positivos. Este contrato se amplía más allá de comprobar nombres de columnas. Compilar verifica sintaxis; las pruebas de comportamiento se realizan tras la importación.


## Ejercicio 7 — Imports

Provoca y captura un ModuleNotFoundError para el paquete del ejercicio anterior. Explica la diferencia entre existir en disco y estar en sys.path. Corrígelo y prueba tanto datos válidos como inválidos.


In [ ]:
# Eliminamos solo las importaciones de este paquete de práctica para repetir el ejemplo.
for nombre in list(sys.modules):
    if nombre == "ventas_practica" or nombre.startswith("ventas_practica."):
        sys.modules.pop(nombre)
ruta_src_practica = str((raiz_practica / "src").resolve())
sys.path[:] = [ruta for ruta in sys.path if ruta != ruta_src_practica]
importlib.invalidate_caches()
try:
    importlib.import_module("ventas_practica")
except ModuleNotFoundError as error:
    if error.name != "ventas_practica":
        raise
    print("Error esperado:", error)
else:
    raise AssertionError("Otro paquete ventas_practica está accesible; usar un entorno limpio.")
sys.path.insert(0, ruta_src_practica)
importlib.invalidate_caches()
from ventas_practica.validation import validar_ventas
from ventas_practica.features import preparar_ventas as preparar_ventas_paquete
from ventas_practica.analysis import resumir_ventas
validar_ventas(datos_demo)
preparados_imports = preparar_ventas_paquete(datos_demo)
resumen_imports = resumir_ventas(preparados_imports)
pd.testing.assert_frame_equal(resumen_imports, resumir_practica(preparar_practica(datos_demo)))
for caso in [datos_demo.drop(columns="precio"), datos_demo.assign(precio=0), datos_demo.assign(precio=np.nan)]:
    try:
        validar_ventas(caso)
    except ValueError as error:
        print("Validación esperada:", error)
    else:
        raise AssertionError("Se aceptaron datos inválidos.")
print(resumen_imports)


### Solución razonada

Python busca módulos en sus rutas de importación, no en cualquier carpeta del disco. sys.path se ajusta aquí para aprender el mecanismo. Para un repositorio instalable, lo adecuado es declarar el paquete en pyproject.toml y usar instalación editable.


## Ejercicio 8 — Colisiones

Crea dos funciones con el mismo nombre dentro de módulos distintos. Muestra que un import directo sobrescribe el nombre local y corrige usando imports de módulos o alias explícitos.


In [ ]:
(paquete_practica / "costes.py").write_text("def calcular(valor):\n    return valor * 0.2\n", encoding="utf-8")
(paquete_practica / "ventas.py").write_text("def calcular(valor):\n    return valor * 1.2\n", encoding="utf-8")
importlib.invalidate_caches()
from ventas_practica.costes import calcular
resultado_coste = calcular(100)
from ventas_practica.ventas import calcular
resultado_venta = calcular(100)
assert resultado_coste == 20 and resultado_venta == 120
from ventas_practica import costes as modulo_costes, ventas as modulo_ventas
assert modulo_costes.calcular(100) == 20
assert modulo_ventas.calcular(100) == 120
from ventas_practica.costes import calcular as calcular_coste
from ventas_practica.ventas import calcular as calcular_venta
assert calcular_coste(100) == 20 and calcular_venta(100) == 120
print("Colisión reproducida y corregida mediante módulos y alias.")


### Solución razonada

El segundo import directo vuelve a asignar calcular en el namespace del notebook. Los módulos y alias permiten conservar ambas funciones y muestran su origen. No hay que recurrir a import * para que ocurra una colisión.


## Ejercicio 9 — Dependencia circular

Crea dos módulos que se importen mutuamente antes de definir sus funciones. Captura el ImportError esperado y elimina el ciclo extrayendo una dependencia común a un tercer módulo.


In [ ]:
circulo_practica = raiz_practica / "src" / "circulo_practica"
circulo_practica.mkdir(exist_ok=True)
(circulo_practica / "__init__.py").write_text("", encoding="utf-8")
(circulo_practica / "a.py").write_text("from .b import funcion_b\ndef funcion_a():\n    return 'A'\n", encoding="utf-8")
(circulo_practica / "b.py").write_text("from .a import funcion_a\ndef funcion_b():\n    return 'B'\n", encoding="utf-8")
for nombre in list(sys.modules):
    if nombre == "circulo_practica" or nombre.startswith("circulo_practica."):
        sys.modules.pop(nombre)
importlib.invalidate_caches()
try:
    importlib.import_module("circulo_practica.a")
except ImportError as error:
    print("Circularidad esperada:", error)
else:
    raise AssertionError("El ciclo no se reprodujo.")
(circulo_practica / "common.py").write_text("def mensaje():\n    return 'base compartida'\n", encoding="utf-8")
(circulo_practica / "a.py").write_text("from .common import mensaje\ndef funcion_a():\n    return 'A: ' + mensaje()\n", encoding="utf-8")
(circulo_practica / "b.py").write_text("from .common import mensaje\ndef funcion_b():\n    return 'B: ' + mensaje()\n", encoding="utf-8")
for nombre in list(sys.modules):
    if nombre == "circulo_practica" or nombre.startswith("circulo_practica."):
        sys.modules.pop(nombre)
importlib.invalidate_caches()
from circulo_practica.a import funcion_a as funcion_a_practica
from circulo_practica.b import funcion_b as funcion_b_practica
assert funcion_a_practica() == "A: base compartida"
assert funcion_b_practica() == "B: base compartida"
print(funcion_a_practica(), funcion_b_practica())


### Solución razonada

El fallo aparece al solicitar una función de un módulo parcialmente inicializado. Tras extraer common, ambos módulos dependen del mismo recurso y dejan de necesitarse mutuamente. Un ciclo no siempre impide importar, pero este ejemplo reproduce el caso que sí falla.


## Ejercicio 10 — __init__.py

Expón validar_ventas, preparar_ventas y resumir_ventas como API del paquete. Comprueba que los imports públicos funcionan.


In [ ]:
fuente_api = """from .validation import validar_ventas
from .features import preparar_ventas
from .analysis import resumir_ventas

__all__ = ["validar_ventas", "preparar_ventas", "resumir_ventas"]
"""
(paquete_practica / "__init__.py").write_text(fuente_api, encoding="utf-8")
import ventas_practica
importlib.invalidate_caches()
importlib.reload(ventas_practica)
ventas_practica.validar_ventas(datos_demo)
resumen_api = ventas_practica.resumir_ventas(ventas_practica.preparar_ventas(datos_demo))
pd.testing.assert_frame_equal(resumen_api, resumen_imports)
assert set(ventas_practica.__all__) == {"validar_ventas", "preparar_ventas", "resumir_ventas"}
print("API pública verificada.")


### Solución razonada

Los imports de __init__.py exponen una interfaz cómoda. __all__ declara qué nombres participan en un import con asterisco; no es un mecanismo de privacidad ni impide importar otros atributos. Conviene mantener ligero el inicializador y evitar operaciones de datos al importar.


## Ejercicio 11 — Arquitectura

Diseña un proyecto que consuma una API, limpie datos, calcule indicadores, genere gráficos y tenga tests. Describe dependencias y cómo ejecutar sin acceder a la API en cada prueba.


### Solución razonada

| Componente | Responsabilidad | Dependencias |
|---|---|---|
| `config.py` | Leer URL, límites y rutas; credenciales desde entorno | Biblioteca estándar |
| `api_client.py` | Consultar la API y manejar errores de transporte | Configuración y cliente HTTP |
| `validation.py` | Comprobar el esquema y reglas de entrada | Contrato de datos |
| `cleaning.py` | Corregir incidencias según una política explícita | Datos recibidos |
| `indicators.py` | Calcular indicadores sin red ni gráficos | Datos validados y preparados |
| `plots.py` | Crear figuras a partir de resultados | Resultados analíticos |
| `pipeline.py` | Componer las etapas con dependencias inyectadas | Funciones de los módulos anteriores |
| `cli.py` | Leer argumentos y llamar al pipeline | Configuración y pipeline |
| `tests/` | Contratos, transformaciones y pruebas de regresión | Fixtures locales y sustitutos del cliente |
| `notebooks/` | Explorar e interpretar | API del paquete |

El cliente devuelve datos; no importa indicadores ni visualización. El pipeline coordina, sin convertirse en la implementación de todas las tareas. Las pruebas utilizan respuestas guardadas o un cliente simulado para evitar red y credenciales. Los tests de integración con la API se ejecutan por separado cuando corresponda.


## Ejercicio 12 — Reto final

Revisa el script monolítico siguiente (más de 40 líneas). Detecta nombres, duplicación, números mágicos y responsabilidades mezcladas. Crea un paquete con validación, limpieza, features, análisis, exportación y pipeline. Conserva el resumen y los CSV originales.


In [ ]:
script_monolitico_reto = 'from pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nr = np.random.default_rng(42)\nd = pd.DataFrame({\n    "id": np.arange(1, 1201),\n    "u": r.integers(1, 8, 1200),\n    "p": r.uniform(5, 150, 1200),\n    "c": r.choice(["web", "tienda", "partner"], 1200),\n    "descuento": r.choice([0.0, 0.1, 0.2], 1200),\n})\nd = pd.concat([d, d.iloc[[0, 1]]], ignore_index=True)\nd = d.drop_duplicates()\nd["importe"] = d["u"] * d["p"]\nd["neto"] = d["importe"] * (1 - d["descuento"])\nd["alta"] = d["neto"] > 500\nw = d.loc[d["c"] == "web"].copy()\nt = d.loc[d["c"] == "tienda"].copy()\nw["importe"] = w["u"] * w["p"]\nw["neto"] = w["importe"] * (1 - w["descuento"])\nt["importe"] = t["u"] * t["p"]\nt["neto"] = t["importe"] * (1 - t["descuento"])\nx = d.groupby("c", as_index=False).agg(\n    operaciones=("id", "size"),\n    importe_neto=("neto", "sum"),\n    pct_alta=("alta", "mean"),\n)\nx = x.rename(columns={"c": "canal"})\nprint(x)\na = Path("salida_monolitica")\na.mkdir(exist_ok=True)\nx.to_csv(a / "resumen.csv", index=False)\nw.to_csv(a / "web.csv", index=False)\nt.to_csv(a / "tienda.csv", index=False)\nfig, ax = plt.subplots(figsize=(7, 4))\nax.bar(x["canal"], x["importe_neto"])\nax.set_title("Importe neto por canal")\nax.set_xlabel("Canal")\nax.set_ylabel("Importe neto")\nplt.show()\nfig, ax = plt.subplots(figsize=(7, 4))\nax.hist(d["neto"], bins=25)\nax.set_title("Distribución de importe neto")\nax.set_xlabel("Importe neto")\nax.set_ylabel("Operaciones")\nplt.show()\n'
print(script_monolitico_reto)
assert len(script_monolitico_reto.splitlines()) >= 40


In [ ]:
# Ejecutar el script inicial en un namespace separado proporciona una referencia.
namespace_reto = {"__name__": "__main__"}
exec(compile(script_monolitico_reto, "<script_monolitico_reto>", "exec"), namespace_reto)
entrada_reto = namespace_reto["d"][["id", "u", "p", "c", "descuento"]].rename(
    columns={"id": "venta_id", "u": "unidades", "p": "precio", "c": "canal"})
# Recuperar también duplicados para comprobar la etapa de limpieza.
entrada_reto = pd.concat([entrada_reto, entrada_reto.iloc[[0, 1]]], ignore_index=True)
entrada_reto_original = entrada_reto.copy(deep=True)
raiz_reto = Path(tempfile.mkdtemp(prefix="reto_", dir=base_practicas)).resolve()
paquete_reto = raiz_reto / "src" / "ventas_reto"
for relativa in ["data/raw", "data/processed", "notebooks", "src/ventas_reto", "tests"]:
    (raiz_reto / relativa).mkdir(parents=True, exist_ok=True)
fuentes_reto = {
    "__init__.py": "from .pipeline import ejecutar\n",
    "validation.py": (paquete_practica / "validation.py").read_text(encoding="utf-8") + """

def validar_reto(datos):
    validar_ventas(datos)
    requeridas = {"venta_id", "descuento"}
    if not requeridas.issubset(datos.columns):
        raise ValueError("Faltan venta_id o descuento.")
    if datos["venta_id"].isna().any():
        raise ValueError("venta_id no puede ser nulo.")
    if not pd.api.types.is_numeric_dtype(datos["descuento"]):
        raise ValueError("El descuento debe ser numérico.")
    if not datos["descuento"].between(0, 1).all():
        raise ValueError("Descuento fuera del intervalo [0, 1].")
""",
    "cleaning.py": """def limpiar(datos):
    return datos.drop_duplicates().copy()
""",
    "features.py": """def preparar(datos, umbral_alta=500):
    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]
    salida["neto"] = salida["importe"] * (1 - salida["descuento"])
    salida["alta"] = salida["neto"] > umbral_alta
    return salida
""",
    "analysis.py": """def resumir(datos):
    return datos.groupby("canal", as_index=False).agg(
        operaciones=("venta_id", "size"), importe_neto=("neto", "sum"), pct_alta=("alta", "mean"))
""",
    "io.py": """from pathlib import Path

def exportar(datos, ruta):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    datos.to_csv(ruta, index=False)
""",
    "pipeline.py": """from .validation import validar_reto
from .cleaning import limpiar
from .features import preparar
from .analysis import resumir

def ejecutar(datos, umbral_alta=500):
    validar_reto(datos)
    preparados = preparar(limpiar(datos), umbral_alta)
    return preparados, resumir(preparados)
""",
}
for nombre, fuente in fuentes_reto.items():
    compile(fuente, nombre, "exec")
    (paquete_reto / nombre).write_text(fuente, encoding="utf-8")
ruta_src_reto = str((raiz_reto / "src").resolve())
sys.path.insert(0, ruta_src_reto)
for nombre in list(sys.modules):
    if nombre == "ventas_reto" or nombre.startswith("ventas_reto."):
        sys.modules.pop(nombre)
importlib.invalidate_caches()
from ventas_reto import ejecutar as ejecutar_reto
from ventas_reto.io import exportar as exportar_reto
preparados_reto, resumen_reto = ejecutar_reto(entrada_reto)
pd.testing.assert_frame_equal(resumen_reto, namespace_reto["x"])
pd.testing.assert_frame_equal(entrada_reto, entrada_reto_original)
np.testing.assert_allclose(preparados_reto["neto"], namespace_reto["d"]["neto"])
assert len(preparados_reto) == 1200
ruta_resumen_reto = raiz_reto / "data/processed/resumen.csv"
exportar_reto(resumen_reto, ruta_resumen_reto)
pd.testing.assert_frame_equal(pd.read_csv(ruta_resumen_reto), resumen_reto)
for canal, ruta in [("web", "web.csv"), ("tienda", "tienda.csv")]:
    exportar_reto(preparados_reto.loc[preparados_reto["canal"] == canal], raiz_reto / "data/processed" / ruta)
# Comparar también las exportaciones por canal, normalizando nombres de columnas.
renombrar_reto = {"id": "venta_id", "u": "unidades", "p": "precio", "c": "canal"}
for archivo in ["web.csv", "tienda.csv"]:
    inicial = pd.read_csv(Path("salida_monolitica") / archivo).rename(columns=renombrar_reto)
    modular = pd.read_csv(raiz_reto / "data/processed" / archivo)
    pd.testing.assert_frame_equal(inicial, modular)
print("Reto verificado: módulos importables, datos equivalentes, tres CSV equivalentes y entrada conservada.")


### Solución razonada

La referencia calcula netos dos veces en grupos web/tienda, mezcla transformación con persistencia y dibuja en el mismo script. El paquete centraliza el cálculo, parametriza el umbral y separa exportación de análisis. Los nombres cambian, pero se conservan los datos y los tres resultados exportados. El notebook sigue siendo el lugar adecuado para interpretar y visualizar. La validación adicional define el contrato; las entradas válidas conservan el comportamiento del script original.


# 17. Preguntas de reflexión


1. ¿Puede un código funcionar y ser de mala calidad?
2. ¿Cuándo un nombre es demasiado corto?
3. ¿DRY significa eliminar cualquier repetición?
4. ¿Qué diferencia hay entre cohesión y acoplamiento?
5. ¿Por qué una función con muchas responsabilidades es difícil de testear?
6. ¿Qué aporta separar `src/` y `notebooks/`?
7. ¿Qué riesgo tiene `import *`?
8. ¿Qué suele indicar una dependencia circular?
9. ¿Crear más archivos garantiza mejor diseño?
10. ¿Qué parte de este notebook debería quedarse como notebook y qué parte debería trasladarse a módulos?

## Respuestas orientativas

1. Sí: puede producir un resultado correcto y tener dependencias ocultas, duplicación o costes de mantenimiento elevados.
2. Cuando omite información que el lector necesita; los nombres cortos pueden ser apropiados en contextos matemáticos o locales claros.
3. No. DRY evita duplicar conocimiento o reglas; no fusiona responsabilidades diferentes por similitud superficial.
4. Cohesión: relación entre tareas de un componente. Acoplamiento: dependencia entre componentes. Se busca cohesión alta y dependencias claras.
5. Porque requiere preparar muchos contextos y mezcla cálculos con efectos como red, impresión o exportación.
6. Permite reutilizar y probar la lógica estable, mientras los notebooks exploran y comunican.
7. Introduce nombres de origen poco visible y facilita colisiones.
8. Puede indicar responsabilidades mal delimitadas o dependencias que conviene extraer o reorganizar.
9. No. Más archivos pueden añadir fragmentación; cada módulo debe tener una responsabilidad coherente.
10. Mantener explicación, gráficos e interpretación en el notebook; trasladar carga, contratos, transformaciones y análisis reusable a módulos.
